# 01 · Foundation — semantic view

**Goal:** join the three tables into a clean base, then build a **semantic view**
Cortex Analyst can answer in natural language.

**Grain rule:** one row per survey response. Each response points to exactly one
game round and one player, so the joins are 1:1 — no fan-out, no double counting.

> Prereq: you ran `00_setup`. If not, do that first.

### Set your context (run every time you open a notebook)

In [ ]:
SET sch = 'PLG_CORTEX_WORKSHOP.WS_' || REGEXP_REPLACE(CURRENT_USER(), '[^A-Za-z0-9_]', '_');
USE WAREHOUSE PLG_WORKSHOP_WH;
USE SCHEMA IDENTIFIER($sch);
SELECT CURRENT_SCHEMA();

### 1. Base view — join the three sources

In [ ]:
CREATE OR REPLACE VIEW SURVEY_BASE AS
SELECT
  s.response_id, s.player_id, s.brand, s.email_type, s.survey_date,
  s.clarity_rating, s.tone_rating, s.clarity_comment, s.tone_comment,
  g.game_round_performance,
  b.replay_rate
FROM SURVEY_RESPONSES s
JOIN GAME_ROUNDS      g ON s.game_round_id = g.game_round_id
JOIN PLAYER_BEHAVIOUR b ON s.player_id     = b.player_id;

-- Confirm the grain didn't change: this should equal the SURVEY_RESPONSES count.
SELECT COUNT(*) AS base_rows FROM SURVEY_BASE;

### 2. Build the semantic view

Most of it is written for you. **Fill the two blanks** marked `>>> YOUR PART <<<`:
1. a dimension for `email_type`, and
2. a metric `pct_low_clarity` = percent of responses with clarity_rating <= 2.

Clause order is fixed: TABLES → RELATIONSHIPS → FACTS → DIMENSIONS → METRICS.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW SURVEY_ANALYSIS
  TABLES (
    responses AS SURVEY_RESPONSES PRIMARY KEY (response_id)
      WITH SYNONYMS ('survey','feedback','email survey')
      COMMENT = 'One row per player email-survey response',
    rounds AS GAME_ROUNDS PRIMARY KEY (game_round_id)
      COMMENT = 'The game round that triggered each survey',
    players AS PLAYER_BEHAVIOUR PRIMARY KEY (player_id)
      COMMENT = 'Player-level behaviour'
  )
  RELATIONSHIPS (
    resp_to_round  AS responses (game_round_id) REFERENCES rounds (game_round_id),
    resp_to_player AS responses (player_id)     REFERENCES players (player_id)
  )
  FACTS (
    responses.clarity_rating AS clarity_rating,
    responses.tone_rating    AS tone_rating,
    players.replay_rate      AS replay_rate
  )
  DIMENSIONS (
    responses.brand AS brand
      WITH SYNONYMS ('lottery brand') COMMENT = 'NPL or VriendenLoterij',
    -- >>> YOUR PART <<< : add an email_type dimension. Give it SAMPLE_VALUES
    --     ('welcome','prize_notification','monthly_update','winback') and IS_ENUM.
    responses.survey_date AS survey_date COMMENT = 'Date the survey was answered',
    rounds.game_round_performance AS game_round_performance
      WITH SYNONYMS ('game outcome','round result')
      COMMENT = 'good / normal / poor'
      SAMPLE_VALUES ('good','normal','poor') IS_ENUM
  )
  METRICS (
    responses.response_count AS COUNT(responses.response_id) COMMENT = 'Number of responses',
    responses.avg_clarity    AS AVG(responses.clarity_rating) COMMENT = 'Average clarity (1-5)',
    responses.avg_tone       AS AVG(responses.tone_rating)    COMMENT = 'Average tone (1-5)'
    -- >>> YOUR PART <<< : add pct_low_clarity =
    --     AVG(IFF(responses.clarity_rating <= 2, 1, 0)) * 100
  )
  COMMENT = 'Player email-survey clarity/tone analysis'
  AI_SQL_GENERATION 'A poor game round means game_round_performance = ''poor''. Low clarity means clarity_rating <= 2. Round averages to 2 decimals.';

_Stuck? The completed statement is in `reference/scaffold.sql` (section 1.2)._

### 3. Self-check ✅ — query the semantic view
These are the same objects Cortex Analyst uses under the hood.

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  SURVEY_ANALYSIS
  METRICS responses.avg_clarity, responses.response_count
  DIMENSIONS responses.brand
) ORDER BY brand;

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  SURVEY_ANALYSIS
  METRICS responses.avg_clarity, responses.pct_low_clarity
  DIMENSIONS rounds.game_round_performance
) ORDER BY game_round_performance;

**Checkpoint:** clarity should be clearly lower for `poor` rounds than `good`
ones. If both queries return sensible numbers, you're done.

Now try it in natural language: open **AI & ML → Cortex Analyst**, pick
`SURVEY_ANALYSIS`, and ask *"average clarity by brand last month"*.

Next: `02_core_ai_precompute.ipynb`.